In [1]:
import polars as pl
df = pl.scan_parquet('./dataset/train.parquet/train.parquet')
train_labels = pl.read_csv('./dataset/train.parquet/train_labels.csv')
unique_customers = df.select("customer_ID").unique().collect()


In [2]:
feat_ids = df.select("customer_ID").unique().collect().get_column("customer_ID")
lab_ids = train_labels.get_column("customer_ID").unique()

missing_labels = feat_ids.filter(~feat_ids.is_in(lab_ids))
missing_features = lab_ids.filter(~lab_ids.is_in(feat_ids))

print(f"IDs in features without labels: {len(missing_labels)}")
print(f"IDs in labels without features: {len(missing_features)}")

if len(missing_labels) == 0 and len(missing_features) == 0:
    print("The customer sets are identical.")

C:\Users\HP\AppData\Local\Temp\ipykernel_18968\2131317591.py:4: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  missing_labels = feat_ids.filter(~feat_ids.is_in(lab_ids))
C:\Users\HP\AppData\Local\Temp\ipykernel_18968\2131317591.py:5: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  missing_features = lab_ids.filter(~lab_ids.is_in(feat_ids))


IDs in features without labels: 0
IDs in labels without features: 0
The customer sets are identical.


In [5]:
print(type(df))
print(type(train_labels))

<class 'polars.lazyframe.frame.LazyFrame'>
<class 'polars.dataframe.frame.DataFrame'>


In [3]:
merged_data = (
    df.lazy().join(train_labels.lazy() if isinstance(train_labels , pl.DataFrame) else train_labels , on = "customer_ID" , how = "left").collect(streaming = True)
)

C:\Users\HP\AppData\Local\Temp\ipykernel_18968\527977169.py:2: DeprecationWarning: the `streaming` parameter was deprecated in 1.25.0; use `engine` instead.
  df.lazy().join(train_labels.lazy() if isinstance(train_labels , pl.DataFrame) else train_labels , on = "customer_ID" , how = "left").collect(streaming = True)


In [4]:
for col, dtype in merged_data.schema.items():
    if dtype == pl.String: 
        print(f"Column: {col} | Type: {dtype}")

Column: customer_ID | Type: String
Column: S_2 | Type: String


In [5]:
numeric_cols = merged_data.select(pl.col(pl.Float32, pl.Float64, pl.Int32, pl.Int16, pl.Int8)).columns
numeric_cols = [c for c in numeric_cols if c not in ['customer_ID', 'target']]

cleaned_data = merged_data.with_columns([
    pl.col(numeric_cols).fill_null(-999)
])


cleaned_data = cleaned_data.with_columns([
    pl.col("S_2").fill_null(pl.date(1900, 1, 1))
])

In [8]:

all_features = [c for c in cleaned_data.columns if c not in ["customer_ID","target","S_2"]]
agg_exprs = [
    *[pl.col(c).mean().alias(f"{c}_mean") for c in all_features],
    *[pl.col(c).last().alias(f"{c}_last") for c in all_features],
    *[(pl.col(c).mean() - pl.col(c).last()).alias(f"{c}_delta") for c in all_features],
]


features_profiles = (
    cleaned_data.lazy()
    .group_by(["customer_ID","target"])
    .agg(agg_exprs)
    .collect(streaming = True)
)

features_profiles.write_parquet("final_data.parquet")

C:\Users\HP\AppData\Local\Temp\ipykernel_18968\927076789.py:13: DeprecationWarning: the `streaming` parameter was deprecated in 1.25.0; use `engine` instead.
  .collect(streaming = True)
